The data wrangling for my capstone project will be in two main parts.  

The first part will involve capturing the company data for the companies that comprise the S&P 500 Dividend Aristocrats index, as well as details for the various index benchmarks that will be used during the analysis.  These benchmarks include the main S&P 500 index, the 11 S&P 500 sector indices, and finally the exchange-traded fund (ETF) that tracks the Dividend Aristocrats index (ticker = NOBL).  

The second part will involve fetching the price, volume, dividend, and stock split data for all of the above.  

The final output will be saved .csv files for the following:
Aristocrats details
Indices and ETF details
Price, volume, dividend, and stock split data

The source for company information for the Aristocrats will be Wikipedia, and the source for price/volume/dividend/split data will be the yfinance library for python.


In [1]:
# Let's start by wrangling the companies that comprise the S&P 500 Dividend Aristocrats
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

# Wikipedia page for S&P 500 Dividend Aristocrats
aristocrats_url = 'https://en.wikipedia.org/wiki/S%26P_500_Dividend_Aristocrats'

# Add a user-agent header to mimic a browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Make the request with headers
response = requests.get(aristocrats_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Create list of tables from Wikipedia page
aristocrats_companies = pd.read_html(StringIO(str(soup)))

# Verify that list of tables are returned
type(aristocrats_companies)

list

In [2]:
# Visually inspect the data scraped from Wikipedia article
print(aristocrats_companies)

[   Ticker symbol                       Company                  Sector
0            AOS                    A.O. Smith             Industrials
1            ABT           Abbott Laboratories             Health Care
2           ABBV                        AbbVie             Health Care
3            AFL                         AFLAC              Financials
4            APD      Air Products & Chemicals               Materials
..           ...                           ...                     ...
64          TROW        T Rowe Price Group Inc              Financials
65           TGT                   Target Corp  Consumer Discretionary
66           GWW                W. W. Grainger             Industrials
67           WMT                  Walmart Inc.        Consumer Staples
68           WST  West Pharmaceutical Services             Health Care

[69 rows x 3 columns],     Year                                              Added  \
0   2025  Erie Indemnity (ERIE), Eversource Energy (ES),... 

In [3]:
# Fetch 1st table from list.
# This table lists the current companies in the index.
# Verify DataFrame from 1st table's contents and visually inspect first few records.
aristocrats = aristocrats_companies[0]
print(type(aristocrats))
print(aristocrats.head(20))

<class 'pandas.core.frame.DataFrame'>
   Ticker symbol                    Company                  Sector
0            AOS                 A.O. Smith             Industrials
1            ABT        Abbott Laboratories             Health Care
2           ABBV                     AbbVie             Health Care
3            AFL                      AFLAC              Financials
4            APD   Air Products & Chemicals               Materials
5            ALB      Albemarle Corporation               Materials
6           AMCR                      Amcor               Materials
7            ADM  Archer-Daniels-Midland Co        Consumer Staples
8            ATO          Atmos Energy Corp               Utilities
9            ADP  Automatic Data Processing  Information Technology
10           BDX      Becton Dickinson & Co             Health Care
11           BRO         Brown & Brown Inc.              Financials
12          BF.B     Brown–Forman (class B)        Consumer Staples
13        

In [4]:
# Change Ticker symbol column header to just Ticker
aristocrats.rename(columns={"Ticker symbol": "Ticker"}, inplace=True)
aristocrats.head()

,Ticker,Company,Sector
0,AOS,A.O. Smith,Industrials
1,ABT,Abbott Laboratories,Health Care
2,ABBV,AbbVie,Health Care
3,AFL,AFLAC,Financials
4,APD,Air Products & Chemicals,Materials


In [5]:
# Fetch 2nd table from list.
# This table is focused on additions and removals of companies over the life of the index.
# Company survivorship of an index is an integral part of the backtesting of any investment strategy.
additions_removals = aristocrats_companies[1]
print(type(additions_removals))
print(additions_removals)

<class 'pandas.core.frame.DataFrame'>
    Year                                              Added  \
0   2025  Erie Indemnity (ERIE), Eversource Energy (ES),...   
1   2024                                Fastenal Co. (FAST)   
2   2023  CH Robinson Worldwide (CHRW), Nordson (NDSN), ...   
3   2022  Brown & Brown, Inc. (BRO) and Church & Dwight ...   
4   2021  IBM (IBM), NextEra Energy (NEE) and West Pharm...   
5   2020  Amcor (AMCR), Atmos Energy (ATO), Realty Incom...   
6   2019  Chubb Limited (CB), People's United Financial ...   
7   2018                                               none   
8   2017                                               none   
9   2016                                               none   
10  2015                                               none   
11  2014                                               none   
12  2013                                               none   
13  2012  AT&T (T), Colgate-Palmolive (CL), Franklin Tem...   
14  2011  Ecolab 

In [6]:
# I intend to start the machine learning with year 2013 as NOBL was launched in 2013;
# therefore, I only require the additions and removals since 2013 (inclusive).
additions_removals = additions_removals[additions_removals['Year'] >= 2013]
additions_removals

,Year,Added,Removed,Reference(s)
0,2025,"Erie Indemnity (ERIE), Eversource Energy (ES),...",none,[7]
1,2024,Fastenal Co. (FAST),Walgreens Boots Alliance Inc. (WBA) and the 3M...,[8][9]
2,2023,"CH Robinson Worldwide (CHRW), Nordson (NDSN), ...",VF Corporation (VFC),[10][11][12]
3,2022,"Brown & Brown, Inc. (BRO) and Church & Dwight ...","AT&T Inc. (T); People's United Financial, Inc....",[13][14][15]
4,2021,"IBM (IBM), NextEra Energy (NEE) and West Pharm...","Raytheon (RTX), Carrier Global (CARR), Otis Wo...",[16]
5,2020,"Amcor (AMCR), Atmos Energy (ATO), Realty Incom...",Ross Stores (ROST) and Helmerich & Payne (HP).,[17]
6,2019,"Chubb Limited (CB), People's United Financial ...",none,[18]
7,2018,none,none,NaN
8,2017,none,none,NaN
9,2016,none,Chubb Corp (CB) (upon acquisition by ACE Limit...,NaN


In [7]:
# Parse the tickers from the 'Added' column and include in new 'Added_Tickers' column
additions_removals = additions_removals.copy()
additions_removals['Added_Tickers'] = additions_removals.loc[:, 'Added'].str.findall(r'\(([A-Z\s]+)\)')

# Parse the tickers from the 'Removed' column and include in new 'Removed_Tickers' column
additions_removals = additions_removals.copy()
additions_removals['Removed_Tickers'] = additions_removals.loc[:, 'Removed'].str.findall(r'\(([A-Z\s]+)\)')
additions_removals

,Year,Added,Removed,Reference(s),Added_Tickers,Removed_Tickers
0,2025,"Erie Indemnity (ERIE), Eversource Energy (ES),...",none,[7],"[ERIE, ES, RGLD, FDS]",[]
1,2024,Fastenal Co. (FAST),Walgreens Boots Alliance Inc. (WBA) and the 3M...,[8][9],[FAST],"[WBA, MMM]"
2,2023,"CH Robinson Worldwide (CHRW), Nordson (NDSN), ...",VF Corporation (VFC),[10][11][12],"[CHRW, NDSN, SJM, KVUE]",[VFC]
3,2022,"Brown & Brown, Inc. (BRO) and Church & Dwight ...","AT&T Inc. (T); People's United Financial, Inc....",[13][14][15],"[BRO, CHD]","[T, PBCT]"
4,2021,"IBM (IBM), NextEra Energy (NEE) and West Pharm...","Raytheon (RTX), Carrier Global (CARR), Otis Wo...",[16],"[IBM, NEE, WST]","[RTX, CARR, OTIS, SYK, LEG]"
5,2020,"Amcor (AMCR), Atmos Energy (ATO), Realty Incom...",Ross Stores (ROST) and Helmerich & Payne (HP).,[17],"[AMCR, ATO, O, ESS, ROST, ALB, EXPD, CARR, OTIS]","[ROST, HP]"
6,2019,"Chubb Limited (CB), People's United Financial ...",none,[18],"[CB, PBCT, CAT, UTX]",[]
7,2018,none,none,NaN,[],[]
8,2017,none,none,NaN,[],[]
9,2016,none,Chubb Corp (CB) (upon acquisition by ACE Limit...,NaN,[],"[CB, ACE]"


In [8]:
# The Added, Removed, and Reference(s) columns are no longer required at this point
additions_removals = additions_removals.drop(columns=['Added', 'Removed', 'Reference(s)'])
additions_removals

,Year,Added_Tickers,Removed_Tickers
0,2025,"[ERIE, ES, RGLD, FDS]",[]
1,2024,[FAST],"[WBA, MMM]"
2,2023,"[CHRW, NDSN, SJM, KVUE]",[VFC]
3,2022,"[BRO, CHD]","[T, PBCT]"
4,2021,"[IBM, NEE, WST]","[RTX, CARR, OTIS, SYK, LEG]"
5,2020,"[AMCR, ATO, O, ESS, ROST, ALB, EXPD, CARR, OTIS]","[ROST, HP]"
6,2019,"[CB, PBCT, CAT, UTX]",[]
7,2018,[],[]
8,2017,[],[]
9,2016,[],"[CB, ACE]"


In [9]:
# Explode the list of tickers in the Added_Tickers column
# Insert the content into new DataFrame along with Year value
additions_filtered = (additions_removals[additions_removals['Added_Tickers'].apply(
    lambda x: isinstance(x, list) and len(x) > 0)]
    .explode('Added_Tickers')
    .rename(columns={'Added_Tickers': 'Ticker', 'Year': 'Year_Added'})
    [['Year_Added', 'Ticker']]
    .reset_index(drop=True))


# Explode the list of tickers in the Removed_Tickers column
# Insert the content into new DataFrame along with Year value
removals_filtered = (additions_removals[additions_removals['Removed_Tickers'].apply(
    lambda x: isinstance(x, list) and len(x) > 0)]
    .explode('Removed_Tickers')
    .rename(columns={'Removed_Tickers': 'Ticker', 'Year': 'Year_Removed'})
    [['Year_Removed', 'Ticker']]
    .reset_index(drop=True))

# Merge the two above DataFrames into a single source of truth
combined_filtered = pd.merge(additions_filtered, 
                            removals_filtered, 
                            on='Ticker', 
                            how='outer')

# Reorder the columns
col_reorder = ['Ticker', 'Year_Added', 'Year_Removed']
combined_filtered = combined_filtered[col_reorder]

# Examine the final result
combined_filtered

,Ticker,Year_Added,Year_Removed
0,ACE,NaN,2016.0
1,ALB,2020.0,NaN
2,AMCR,2020.0,NaN
3,ATO,2020.0,NaN
4,BMS,NaN,2014.0
5,BRO,2022.0,NaN
6,CARR,2020.0,2021.0
7,CAT,2019.0,NaN
8,CB,2019.0,2016.0
9,CHD,2022.0,NaN


In [10]:
# Since I am having all additions and removals occur on January 1st,
# any ticker that is added and removed in same year should be omitted.
combined_filtered = combined_filtered[combined_filtered['Year_Added'] != combined_filtered['Year_Removed']]
combined_filtered.reset_index(drop=True, inplace=True) 
combined_filtered

,Ticker,Year_Added,Year_Removed
0,ACE,NaN,2016.0
1,ALB,2020.0,NaN
2,AMCR,2020.0,NaN
3,ATO,2020.0,NaN
4,BMS,NaN,2014.0
5,BRO,2022.0,NaN
6,CARR,2020.0,2021.0
7,CAT,2019.0,NaN
8,CB,2019.0,2016.0
9,CHD,2022.0,NaN


In [11]:
# Merge the aristocrats and combined_filters DataFrames
aristocrats_complete = pd.merge(aristocrats, 
                            combined_filtered, 
                            on='Ticker', 
                            how='outer')
aristocrats_complete.head(20)

,Ticker,Company,Sector,Year_Added,Year_Removed
0,ABBV,AbbVie,Health Care,NaN,NaN
1,ABT,Abbott Laboratories,Health Care,NaN,NaN
2,ACE,NaN,NaN,NaN,2016.0
3,ADM,Archer-Daniels-Midland Co,Consumer Staples,NaN,NaN
4,ADP,Automatic Data Processing,Information Technology,NaN,NaN
5,AFL,AFLAC,Financials,NaN,NaN
6,ALB,Albemarle Corporation,Materials,2020.0,NaN
7,AMCR,Amcor,Materials,2020.0,NaN
8,AOS,A.O. Smith,Industrials,NaN,NaN
9,APD,Air Products & Chemicals,Materials,NaN,NaN


In [12]:
# For those records where Year_Added is NaN, insert 2013 (start date for
# time series machine learning)
aristocrats_complete['Year_Added'] = aristocrats_complete['Year_Added'].fillna(2013)

# For those records where Year_Removed is NaN, insert 2030 (arbitrary year
# in near future)
aristocrats_complete['Year_Removed'] = aristocrats_complete['Year_Removed'].fillna(2030)

aristocrats_complete.head(20)

,Ticker,Company,Sector,Year_Added,Year_Removed
0,ABBV,AbbVie,Health Care,2013.0,2030.0
1,ABT,Abbott Laboratories,Health Care,2013.0,2030.0
2,ACE,NaN,NaN,2013.0,2016.0
3,ADM,Archer-Daniels-Midland Co,Consumer Staples,2013.0,2030.0
4,ADP,Automatic Data Processing,Information Technology,2013.0,2030.0
5,AFL,AFLAC,Financials,2013.0,2030.0
6,ALB,Albemarle Corporation,Materials,2020.0,2030.0
7,AMCR,Amcor,Materials,2020.0,2030.0
8,AOS,A.O. Smith,Industrials,2013.0,2030.0
9,APD,Air Products & Chemicals,Materials,2013.0,2030.0


**Note:**  Yfinance does not currently accept the period (.) delimiter to distinguish share class for the ticker symbol, and any call to yfinance using a ticker that has this delimiter (e.g., "BF.B") will fail.  Yfinance utilizes the hyphen char (-) instead.  

In [13]:
# Identify any tickers that have period (.) char as substring and  
# replace with hyphen (-) char.
aristocrats_complete['Ticker'] = aristocrats_complete['Ticker'].str.\
                                    replace('.', '-', regex=False)
aristocrats_complete.head(20)

,Ticker,Company,Sector,Year_Added,Year_Removed
0,ABBV,AbbVie,Health Care,2013.0,2030.0
1,ABT,Abbott Laboratories,Health Care,2013.0,2030.0
2,ACE,NaN,NaN,2013.0,2016.0
3,ADM,Archer-Daniels-Midland Co,Consumer Staples,2013.0,2030.0
4,ADP,Automatic Data Processing,Information Technology,2013.0,2030.0
5,AFL,AFLAC,Financials,2013.0,2030.0
6,ALB,Albemarle Corporation,Materials,2020.0,2030.0
7,AMCR,Amcor,Materials,2020.0,2030.0
8,AOS,A.O. Smith,Industrials,2013.0,2030.0
9,APD,Air Products & Chemicals,Materials,2013.0,2030.0


In [14]:
# There are several tickers in the DataFrame with missing Company and Sector. 
# Use yfinance to fetch these values.  I suspect that some of these tickers
# will return Company=None and Sector=None as they have since been delisted
# from the exchanges.
import yfinance as yf
import time

for index, row in aristocrats_complete.iterrows():
    if not pd.isna(row['Company']):
        continue
    try:
        info = yf.Ticker(row['Ticker']).info
        aristocrats_complete.at[index, 'Sector'] = info.get('sector')
        aristocrats_complete.at[index, 'Company'] = info.get('longName')
        time.sleep(0.5)
    except Exception as e:
        print(f"Error fetching {row['Ticker']}: {e}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


**Note:** The 404 error returned for WBA indicates that WBA is the first of a possible handful of symbols that have been delisted from the exchanges.  I'll need to check further on this.

In [15]:
# Quick look at the results
print(aristocrats_complete.head(15))
print(aristocrats_complete.tail(15))

   Ticker                    Company                  Sector  Year_Added  \
0    ABBV                     AbbVie             Health Care      2013.0   
1     ABT        Abbott Laboratories             Health Care      2013.0   
2     ACE                       None                    None      2013.0   
3     ADM  Archer-Daniels-Midland Co        Consumer Staples      2013.0   
4     ADP  Automatic Data Processing  Information Technology      2013.0   
5     AFL                      AFLAC              Financials      2013.0   
6     ALB      Albemarle Corporation               Materials      2020.0   
7    AMCR                      Amcor               Materials      2020.0   
8     AOS                 A.O. Smith             Industrials      2013.0   
9     APD   Air Products & Chemicals               Materials      2013.0   
10    ATO          Atmos Energy Corp               Utilities      2020.0   
11    BDX      Becton Dickinson & Co             Health Care      2013.0   
12    BEN   

In [16]:
# Return those tickers that I suspect as being delisted.
# I suspect Sector=None will be returned for all.
delisted = aristocrats_complete[aristocrats_complete['Company'].isna()]
delisted

,Ticker,Company,Sector,Year_Added,Year_Removed
2,ACE,None,None,2013.0,2016.0
36,FDO,None,None,2013.0,2015.0
62,PBCT,None,None,2019.0,2022.0
72,SIAL,None,None,2013.0,2015.0
81,UTX,None,None,2019.0,2030.0
83,WBA,None,None,2013.0,2024.0


In [17]:
# Need to confirm my suspicions by attempting to fetch price data for these
for symbol in delisted['Ticker']:
    price_hist = yf.Ticker(symbol).history(period='max', auto_adjust=False)
    len(price_hist)

$ACE: possibly delisted; no price data found  (1d 1927-07-13 -> 2026-06-18)
$FDO: possibly delisted; no price data found  (1d 1927-07-13 -> 2026-06-18)
$PBCT: possibly delisted; no timezone found
$SIAL: possibly delisted; no price data found  (1d 1927-07-13 -> 2026-06-18)
$UTX: possibly delisted; no timezone found
$WBA: possibly delisted; no timezone found


In [18]:
# Remove the delisted stocks as yfinance does not have price data for these
aristocrats_complete.drop(delisted.index, inplace=True)
aristocrats_complete.reset_index(drop=True, inplace=True) 
aristocrats_complete.tail(15)

,Ticker,Company,Sector,Year_Added,Year_Removed
66,ROP,Roper Technologies,Industrials,2013.0,2030.0
67,RTX,RTX Corporation,Industrials,2013.0,2021.0
68,SHW,Sherwin-Williams,Materials,2013.0,2030.0
69,SJM,The J. M. Smucker Company,Consumer Staples,2023.0,2030.0
70,SPGI,S&P Global Inc,Financials,2013.0,2030.0
71,SWK,Stanley Black & Decker,Industrials,2013.0,2030.0
72,SYK,Stryker Corporation,Healthcare,2013.0,2021.0
73,SYY,Sysco Corp,Consumer Staples,2013.0,2030.0
74,T,AT&T Inc.,Communication Services,2013.0,2022.0
75,TGT,Target Corp,Consumer Discretionary,2013.0,2030.0


In [19]:
# Determine any records where Sector value is missing
aristocrats_complete[aristocrats_complete['Sector'].isna()]

,Ticker,Company,Sector,Year_Added,Year_Removed
13,BMS,"Bemis Company, Inc.",None,2013.0,2014.0


In [20]:
# Some research on Google showed that Bemis Company Inc. was the Materials 
# sector of the economy
aristocrats_complete.loc[aristocrats_complete['Ticker'] == 'BMS', 'Sector'] = 'Materials'

In [21]:
# The above visual inspection shows that some of the Sector values appear to be 
# different depending from where they were fetched (Wikipedia versus yfinance).  
# Need to double check this.
aristocrats_complete['Sector'].unique()

array(['Health Care', 'Consumer Staples', 'Information Technology',
       'Financials', 'Materials', 'Industrials', 'Utilities', 'Energy',
       'Real Estate', 'Consumer Discretionary', 'Consumer Cyclical',
       'Basic Materials', 'Healthcare', 'Communication Services'],
      dtype=object)

In [22]:
# For those Sector verbiages showing discrepancies, realign using intended verbiages. 
# These same verbiages will be used when creating the dictionary of indices further
# down in this Notebook.
aristocrats_complete['Sector'] = aristocrats_complete['Sector'].replace('Healthcare', 'Health Care')
aristocrats_complete['Sector'] = aristocrats_complete['Sector'].replace('Basic Materials', 'Materials')
aristocrats_complete['Sector'] = aristocrats_complete['Sector'].replace('Consumer Cyclical', 'Consumer Discretionary')
aristocrats_complete['Sector'].unique()

# Quick visual inspection
aristocrats_complete.head(15)
aristocrats_complete.tail(15)

,Ticker,Company,Sector,Year_Added,Year_Removed
66,ROP,Roper Technologies,Industrials,2013.0,2030.0
67,RTX,RTX Corporation,Industrials,2013.0,2021.0
68,SHW,Sherwin-Williams,Materials,2013.0,2030.0
69,SJM,The J. M. Smucker Company,Consumer Staples,2023.0,2030.0
70,SPGI,S&P Global Inc,Financials,2013.0,2030.0
71,SWK,Stanley Black & Decker,Industrials,2013.0,2030.0
72,SYK,Stryker Corporation,Health Care,2013.0,2021.0
73,SYY,Sysco Corp,Consumer Staples,2013.0,2030.0
74,T,AT&T Inc.,Communication Services,2013.0,2022.0
75,TGT,Target Corp,Consumer Discretionary,2013.0,2030.0


In [24]:
# Iterate through the aristocrats_complete DataFrame and fetch
# historical price and dividend data since 1/1/2013.
stock_symbols = aristocrats_complete['Ticker']
stock_data = {
    symbol: yf.Ticker(symbol).history(start='2013-01-01', auto_adjust=False) for symbol in stock_symbols
}
print("Done")

# Quick look at dictionary of market data
print(stock_data)

Done
{'ABBV':                                  Open        High         Low       Close  \
Date                                                                        
2013-01-02 00:00:00-05:00   34.919998   35.400002   34.099998   35.119999   
2013-01-03 00:00:00-05:00   35.000000   35.000000   34.160000   34.830002   
2013-01-04 00:00:00-05:00   34.619999   34.889999   34.250000   34.389999   
2013-01-07 00:00:00-05:00   34.150002   35.450001   34.150002   34.459999   
2013-01-08 00:00:00-05:00   34.290001   34.639999   33.360001   33.709999   
...                               ...         ...         ...         ...   
2026-06-12 00:00:00-04:00  227.500000  228.399994  224.320007  227.729996   
2026-06-15 00:00:00-04:00  226.000000  226.500000  220.000000  221.589996   
2026-06-16 00:00:00-04:00  223.309998  223.500000  219.149994  222.470001   
2026-06-17 00:00:00-04:00  222.399994  222.839996  218.830002  221.229996   
2026-06-18 00:00:00-04:00  221.369995  222.300003  215.369995 

In [25]:
# Unfortunately yfinance doesn't have a way to return the S&P 500 
# sector indices as a list; therefore, I will need to manually create 
# a dictionary for the relevant general market and sector indices, 
# as well as the NOBL exchange-traded fund (ETF) benchark that will be 
# used for the time series machine learning.
indices_dict = {
    'Ticker': ['NOBL', 
                      '^GSPC',
					  '^SP500-15', 
					  '^SP500-50', 
					  '^GSPE', 
					  '^SP500-40', 
					  '^SP500-20', 
					  '^SP500-45', 
					  '^SP500-30', 
					  '^SP500-60',
					  '^SP500-55',
					  '^SP500-35',
					  '^SP500-25'],
    'Name': ['S&P 500 Dividend Aristocrats ETF',
             'S&P 500 Index',
	         'S&P 500 Materials Sector Index',
			 'S&P 500 Communication Sector Index',
			 'S&P 500 Energy Sector Index',
			 'S&P 500 Financials Sector Index',
			 'S&P 500 Industrial Sector Index',
			 'S&P 500 Technology Sector Index',
			 'S&P 500 Consumer Staples Sector Index',
			 'S&P 500 Real Estate Sector Index',
			 'S&P 500 Utilities Sector Index',
			 'S&P 500 Health Care Sector Index',
			 'S&P 500 Consumer Discretionary Sector Index'],
    'Type': ['ETF',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index',
             'Index'],
    'Sector': ['NA', 
               'NA',
	           'Materials', 
			   'Communication Services',
			   'Energy',
			   'Financials',
			   'Industrials',
			   'Information Technology',
			   'Consumer Staples',
			   'Real Estate',
			   'Utilities',
			   'Health Care',
			   'Consumer Discretionary']
}

# Convert to DataFrame
indices = pd.DataFrame(indices_dict)
indices

,Ticker,Name,Type,Sector
0,NOBL,S&P 500 Dividend Aristocrats ETF,ETF,NA
1,^GSPC,S&P 500 Index,Index,NA
2,^SP500-15,S&P 500 Materials Sector Index,Index,Materials
3,^SP500-50,S&P 500 Communication Sector Index,Index,Communication Services
4,^GSPE,S&P 500 Energy Sector Index,Index,Energy
5,^SP500-40,S&P 500 Financials Sector Index,Index,Financials
6,^SP500-20,S&P 500 Industrial Sector Index,Index,Industrials
7,^SP500-45,S&P 500 Technology Sector Index,Index,Information Technology
8,^SP500-30,S&P 500 Consumer Staples Sector Index,Index,Consumer Staples
9,^SP500-60,S&P 500 Real Estate Sector Index,Index,Real Estate


In [26]:
# Iterate through the aristocrats_complete DataFrame and fetch
# historical price and dividend data since 1/1/2013.
index_symbols = indices['Ticker']
index_data = {
    symbol: yf.Ticker(symbol).history(start='2013-01-01', auto_adjust=False) for symbol in index_symbols
}
print("Done")

# Quick look at dictionary of index data
print(index_data)

Done
{'NOBL':                                 Open       High        Low      Close  \
Date                                                                    
2013-10-10 00:00:00-04:00  20.200001  20.459999  20.200001  20.375000   
2013-10-11 00:00:00-04:00  20.745001  20.745001  20.350000  20.520000   
2013-10-14 00:00:00-04:00  20.424999  20.605000  20.405001  20.590000   
2013-10-15 00:00:00-04:00  20.555000  20.559999  20.415001  20.434999   
2013-10-16 00:00:00-04:00  20.605000  20.660000  20.495001  20.639999   
...                              ...        ...        ...        ...   
2026-06-12 00:00:00-04:00  55.500000  55.750000  55.310001  55.630001   
2026-06-15 00:00:00-04:00  55.730000  55.880001  55.549999  55.590000   
2026-06-16 00:00:00-04:00  55.740002  56.080002  55.610001  55.820000   
2026-06-17 00:00:00-04:00  55.799999  55.840000  54.740002  54.869999   
2026-06-18 00:00:00-04:00  55.169998  55.220001  54.980000  55.029999   

                           Adj Close

In [27]:
# Create function to standardize price and dividend data for all asset classes
# used in the analysis (stocks, ETFs, and indexes).
def standardize_data(data_dict):
    standardized = []
    for symbol, df in data_dict.items():
        if not df.empty:
            df = df[["Open", "High", "Low", "Close", "Adj Close", "Volume", "Dividends", "Stock Splits"]]
            df = df.dropna()
            for col in df.columns:
                if df[col].dtype == 'object':
                    df[col] = df[col].astype(str).str.replace(r'[^\d.-]', '', regex=True)
            df = df.astype(float)
            df["Symbol"] = symbol
            standardized.append(df)
    return standardized

# Standardize the data from both DataFrames, then concatenate.
market_data = pd.concat(
    standardize_data(stock_data) + standardize_data(index_data)
).reset_index()

# Quick sanity check of resulted DataFrame
print(len(market_data))
print(market_data.head())
print(market_data.tail())

310037
                       Date       Open       High        Low      Close  \
0 2013-01-02 00:00:00-05:00  34.919998  35.400002  34.099998  35.119999   
1 2013-01-03 00:00:00-05:00  35.000000  35.000000  34.160000  34.830002   
2 2013-01-04 00:00:00-05:00  34.619999  34.889999  34.250000  34.389999   
3 2013-01-07 00:00:00-05:00  34.150002  35.450001  34.150002  34.459999   
4 2013-01-08 00:00:00-05:00  34.290001  34.639999  33.360001  33.709999   

   Adj Close      Volume  Dividends  Stock Splits Symbol  
0  20.561075  13767900.0        0.0           0.0   ABBV  
1  20.391291  16739300.0        0.0           0.0   ABBV  
2  20.133686  21372100.0        0.0           0.0   ABBV  
3  20.174667  17897100.0        0.0           0.0   ABBV  
4  19.735580  17863300.0        0.0           0.0   ABBV  
                            Date         Open         High          Low  \
310032 2026-06-09 00:00:00-04:00  1901.479980  1924.859985  1862.640015   
310033 2026-06-10 00:00:00-04:00  1884

In [28]:
# Normalize the values in the Date column to ensure compatibility with future
# pandas operations for time series.
market_data['Date'] = pd.to_datetime(market_data['Date']).dt.normalize()
market_data.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Symbol
0,2013-01-02 00:00:00-05:00,34.919998,35.400002,34.099998,35.119999,20.561075,13767900.0,0.0,0.0,ABBV
1,2013-01-03 00:00:00-05:00,35.000000,35.000000,34.160000,34.830002,20.391291,16739300.0,0.0,0.0,ABBV
2,2013-01-04 00:00:00-05:00,34.619999,34.889999,34.250000,34.389999,20.133686,21372100.0,0.0,0.0,ABBV
3,2013-01-07 00:00:00-05:00,34.150002,35.450001,34.150002,34.459999,20.174667,17897100.0,0.0,0.0,ABBV
4,2013-01-08 00:00:00-05:00,34.290001,34.639999,33.360001,33.709999,19.735580,17863300.0,0.0,0.0,ABBV


In [29]:
# Check for NaN, None, or NaT values
ohlcv = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
nan_data = market_data[market_data[ohlcv].isna().any(axis=1)]
print(len(nan_data))
print(nan_data.head())
print(nan_data.tail())

0
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits, Symbol]
Index: []
Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits, Symbol]
Index: []


In [30]:
# Check zero (0) values for Open, High, Low, Close and Adj Close for the indices.
# Zero volume is immaterial since I only need price data for the
# indices for my analysis.
ohlc = ["Open", "High", "Low", "Close", "Adj Close"]
data_indices = market_data[market_data['Symbol'].str.startswith('^')]
zero_data_indices = data_indices[(data_indices[ohlc] == 0).any(axis=1)]
print(len(zero_data_indices))

0


In [31]:
# Check zero (0) values for Open, High, Low, Close, Adj Close, and Volume for stocks
# and NOBL.  Volume is important here since I will incorporate volume data
# during my analysis.
ohlcv = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
data_stocks= market_data[~market_data['Symbol'].str.startswith('^')]
zero_data_stocks = data_stocks[(data_stocks[ohlcv] == 0).any(axis=1)]
print(len(zero_data_stocks))

1225


In [32]:
# Let's break this number down by ticker symbols
print(zero_data_stocks['Symbol'].value_counts())

Symbol
AMCR    1223
CHD        1
ERIE       1
Name: count, dtype: int64


In [33]:
# Since there are only singleton recoreds for CHD and ERIE, let's look at these first
zero_data_stocks[zero_data_stocks['Symbol'].isin(['CHD', 'ERIE'])]

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Symbol
62514,2019-12-31 00:00:00-05:00,70.349998,70.349998,70.349998,70.349998,65.112663,0.0,0.0,0.0,CHD
98239,2013-12-13 00:00:00-05:00,70.279999,70.279999,70.279999,70.279999,51.991570,0.0,0.0,0.0,ERIE


For CHD there was zero volume for 12/31/2019, but the company was first added to the index in 2022.  For ERIE there was zero volume for 12/13/2013, but ERIE was added in 2025.  In light of this, these records can be deleted from the DataFrame.

In [34]:
# Now let's check the details for AMCR
zero_data_amcr = zero_data_stocks[zero_data_stocks['Symbol'] == 'AMCR']
print(zero_data_amcr.head())
print(zero_data_amcr.tail())

                           Date       Open       High        Low      Close  \
20316 2013-01-02 00:00:00-05:00  41.849998  41.849998  41.849998  41.849998   
20317 2013-01-03 00:00:00-05:00  41.849998  41.849998  41.849998  41.849998   
20318 2013-01-04 00:00:00-05:00  41.849998  41.849998  41.849998  41.849998   
20320 2013-01-08 00:00:00-05:00  43.299999  43.299999  43.299999  43.299999   
20321 2013-01-09 00:00:00-05:00  43.299999  43.299999  43.299999  43.299999   

       Adj Close  Volume  Dividends  Stock Splits Symbol  
20316  24.865166     0.0        0.0           0.0   AMCR  
20317  24.865166     0.0        0.0           0.0   AMCR  
20318  24.865166     0.0        0.0           0.0   AMCR  
20320  25.726679     0.0        0.0           0.0   AMCR  
20321  25.726679     0.0        0.0           0.0   AMCR  
                           Date   Open   High    Low  Close  Adj Close  \
21931 2019-06-04 00:00:00-04:00  55.75  55.75  55.75  55.75   41.19981   
21932 2019-06-05 00:00:

The very last record is in 2019.  AMCR was not added to the index until 2020, so it will be OK to delete these records.

In [35]:
# Drop zero (0) records for AMCR, CHD, and ERIE
market_data.drop(zero_data_stocks.index, inplace=True)
market_data.reset_index(drop=True, inplace=True) 
print(len(market_data))

308812


In [36]:
# Check for the existence of duped records
duplicate_count = market_data.duplicated(subset=['Date', 'Symbol']).sum()
print(duplicate_count)

0


In [37]:
# Save 3 DataFrames to respective .csv files
market_data.to_csv('market_data_for_time_series.csv', index=False)
aristocrats_complete.to_csv('aristocrats_complete_for_time_series.csv', index=False)
indices.to_csv('indices_for_time_series.csv', index=False)

In [38]:
stock_splits = market_data[market_data['Stock Splits'] > 0.0]\
                .sort_values(by='Date', ascending=True)
print(len(stock_splits))
print(stock_splits.tail())

# stock_splits.sort_values(by='Date', ascending=True)

37
                            Date        Open        High         Low  \
79402  2024-09-12 00:00:00-04:00  205.500000  207.490005  201.919998   
113434 2025-05-22 00:00:00-04:00   40.860001   41.119999   40.139999   
22372  2026-01-15 00:00:00-05:00   44.560001   45.490002   43.380001   
35933  2026-02-10 00:00:00-05:00  165.979996  172.580002  164.410004   
268218 2026-05-28 00:00:00-04:00   53.980000   54.139999   53.660000   

             Close   Adj Close     Volume  Dividends  Stock Splits Symbol  
79402   206.020004  202.960648  1340400.0        0.0         4.000   CTAS  
113434   40.720001   39.884617  4420400.0        0.0         2.000   FAST  
22372    44.160000   42.873783  4352500.0        0.0         0.200   AMCR  
35933   171.679993  169.396317  4604400.0        0.0         1.272    BDX  
268218   53.990002   53.990002   851600.0        0.0         2.000   NOBL  


In [39]:
# stock_splits.groupby('Symbol').count()
stock_splits[stock_splits['Symbol']=='FAST'][['Date','Stock Splits']]

,Date,Stock Splits
111926,2019-05-23 00:00:00-04:00,2.0
113434,2025-05-22 00:00:00-04:00,2.0
